### Inference

In [ ]:
import os
import sys

sys.path.append("../../")

from dotenv import load_dotenv
load_dotenv()

from datetime import datetime

import datasets
import shortuuid
import pathlib
import torch
import transformers
from transformers import AutoProcessor
from torch.utils.data import DataLoader
import PIL
import numpy as np

import fastsae
from fastsae.utils.hub import load_from_hub

DATASET_PATH = "evanarlian/imagenet_1k_resized_256"

print(f"Available GPUs: {os.environ.get('CUDA_VISIBLE_DEVICES', 'All')}")
print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch CUDA device count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU name: {torch.cuda.get_device_name()}")

In [ ]:
######################
# config
######################

dataset_name = "evanarlian/imagenet_1k_resized_256"
collate = "auto"
device = "cuda"

# dataloader
_caching = False

# verbose
_verbose = True

### Model Wrapper

In [ ]:
## Option 1: Download from HuggingFace Hub
wrapper, ckpt_path = load_from_hub(
    repo_id="hyesulim/fastsae-models",
    cache_dir="./hf-fastsae-models", # If you want to specify a different cache directory
    return_ckpt_path=True,
)

## Option 2: Use local path
# ckpt_path = "YOUR_PATH"
# backbone = fastsae.models.Backbone.load(os.path.join(ckpt_path, "backbone"))
# sae = fastsae.models.SAE.load(os.path.join(ckpt_path, "final"))
# wrapper = fastsae.models.SAEWrapper(backbone=backbone, sae=sae)

In [ ]:
from rich import print as rprint, inspect
rprint(wrapper)

### Interpreter

In [ ]:
def get_sae_stats_path(sae_load_dir):
    sae_stats_path = os.path.join(sae_load_dir, "stats")
    try:
        candidates = [
            d for d in os.listdir(sae_load_dir)
            if d.startswith("stats") and os.path.isdir(os.path.join(sae_load_dir, d))
        ]
        if len(candidates) > 0:
            # Prefer most recently modified candidate
            candidates_full = [os.path.join(sae_load_dir, d) for d in candidates]
            sae_stats_path = max(candidates_full, key=lambda p: os.path.getmtime(p))
    except Exception:
        # Fallback to default "stats" directory if listing fails
        pass
    return sae_stats_path

stats_path = get_sae_stats_path(os.path.join(ckpt_path, "final"))
print(f"stats_path: {stats_path}")

try:
    intrp_dataset_split = stats_path.split("/")[-1].split(".")[1].split("_")[1]
    intrp_dataset_seed = stats_path.split("/")[-1].split(".")[2].split("_")[1]
except:
    intrp_dataset_split = "val"
    intrp_dataset_seed = "no"

print(f"{intrp_dataset_split=}")
print(f"{intrp_dataset_seed=}")

In [ ]:
# let's use validation split to keep it light!
if intrp_dataset_split == "train":
    intrp_dataset_split = "val"
    stats_path = stats_path.replace("split_train", intrp_dataset_split)
    print(f"stats_path: {stats_path}")

In [ ]:
dataset_instance = datasets.load_dataset(
    path=dataset_name,
    split=intrp_dataset_split,
)
if intrp_dataset_seed != "no":
    dataset_instance = dataset_instance.shuffle(seed=int(intrp_dataset_seed))
auto_processor = transformers.AutoProcessor.from_pretrained(wrapper.backbone.config["model_hf_name"])

dataloader = DataLoader(dataset_instance, batch_size=256, shuffle=False)

In [ ]:
intrp = fastsae.interpret.SAEInterpreter(
    wrapped=wrapper,
    backbone_processor=auto_processor,
    stats_dataloader=dataloader,
)

intrp.load_stats(stats_path)
print(intrp.stats.keys())

> dict_keys(['mean_act', 'sparsity', 'sparsity_thr', 'top_act', 'top_idx', 'top_label', 'top_entropy', 'cls_wise_top_idx', 'metrics'])



### Visualize latents

In [ ]:
intrp.show_stats_scatter_plot()

If you SAE loaded from Huggingface Hub "hyesulim/fastsae-models" you will see this as the cell output:

In [ ]:
# PIL.Image.open("output_examples/Golden-Gate-Bridge_scatter_plot.png")
PIL.Image.open("output_examples/val_scatter.png")

In [ ]:
input_img = "input_examples/Golden-Gate-Bridge.jpg"
# input_img = "input_examples/christmas_socks.jpg"
# Try different images!

input_img = PIL.Image.open(input_img)
out_dict = intrp.inference(input_img)

for k, v in out_dict.items():
    if k.startswith("metrics"):
        print(k, v)

sae_act_token_level = out_dict["sae_act"].squeeze(0).cpu()
sae_act_image_level = sae_act_token_level.mean(dim=0)

sorted_idx_image_level = sae_act_image_level.argsort(descending=True)

If you SAE loaded from Huggingface Hub "hyesulim/fastsae-models" you will see this as the cell output:

<!-- > metrics/z_recon_loss 0.047765668481588364 <br>
> metrics/sae_act_sparsity 0.995212197303772 <br>
> metrics/output_recon_loss 0.05456007272005081 -->

> metrics/z_recon_loss 0.04358726367354393 <br>
> metrics/sae_act_sparsity 0.9967320561408997 <br>
> metrics/output_recon_loss 0.04965989664196968

In [ ]:
intrp.show_sae_activation_bar(sae_act_image_level)

If you SAE loaded from Huggingface Hub "hyesulim/fastsae-models" you will see this as the cell output:

In [ ]:
PIL.Image.open("output_examples/Golden-Gate-Bridge_act_plot.png")

In [ ]:
latent_idx = sorted_idx_image_level[2]
intrp.show_ref_samples(latent_idx, input_img=input_img)

If you SAE loaded from Huggingface Hub "hyesulim/fastsae-models" you will see this as the cell output:

In [ ]:
PIL.Image.open("output_examples/Golden-Gate-Bridge_5000.png")

### Classification with steering

In [ ]:
dataloader_eval = fastsae.dataset.flexible.create_dataloader(
    dataset=dataset_instance, 
    processor=auto_processor,
    return_pixel_values_only=False, # NOTE: Keep this False for classification
    return_label=True, # NOTE: Keep this True for classification to compute accuracy
    batch_size=256,
    shuffle=False,
    collate="auto",  
)

In [ ]:
dataloader_iter = iter(dataloader_eval)
x = next(dataloader_iter)

for k, v in x.items():
    print(k, v.shape)
    assert v.shape[0] == dataloader_eval.batch_size

If you SAE loaded from Huggingface Hub "hyesulim/fastsae-models" you will see this as the cell output:

> input_ids torch.Size([256, 4]) <br>
> attention_mask torch.Size([256, 4]) <br>
> pixel_values torch.Size([256, 3, 224, 224]) <br>
> label torch.Size([256])

In [ ]:
from torch.nn.utils.rnn import pad_sequence
from fastsae.downstream_tasks.openai_imagenet_templates import openai_imagenet_template

def get_text_features_openai(classnames):
    """Calculate mean text features across templates for each class."""
    mean_text_features = 0

    for template_fn in openai_imagenet_template:
        # Generate prompts and convert to token IDs
        prompts = [template_fn(c) for c in classnames]
        prompt_ids = [
            auto_processor(
                text=p, return_tensors="pt", padding=False, truncation=True
            ).input_ids[0]
            for p in prompts
        ]

        # Process batch
        padded_prompts = pad_sequence(prompt_ids, batch_first=True).to(
            wrapper.device
        )

        # Get text features
        with torch.no_grad():
            text_features = wrapper.backbone.model.get_text_features(padded_prompts)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)
            mean_text_features += text_features

    return mean_text_features / len(openai_imagenet_template)


def get_predictions(out, text_features):
    image_features = out.image_embeds
    logit_scale = wrapper.backbone.model.logit_scale.exp()
    logits = logit_scale * (image_features @ text_features.T)
    preds = logits.argmax(dim=-1)
    return preds


filename = "../../fastsae-lib/downstream_tasks/imagenet_classnames.txt"
with open(filename, "r") as file:
    classnames = [" ".join(line.strip().split(" ")[1:]) for line in file.readlines()]

text_features = get_text_features_openai(classnames)

In [ ]:
steer_cfg = None
out_pre, out_post, a = wrapper(x, steer_cfg=steer_cfg)

preds_pre = get_predictions(out_pre, text_features).cpu()
preds_post = get_predictions(out_post, text_features).cpu()

acc_pre = (preds_pre == x["label"]).float().mean()
acc_post = (preds_post == x["label"]).float().mean()

print(acc_pre, acc_post)

If you SAE loaded from Huggingface Hub "hyesulim/fastsae-models" you will see this as the cell output:

> tensor(0.8477) tensor(0.7930)


In [ ]:
steer_cfg = {"indices": range(wrapper.sae.d_sae), "set": 0} # setting all latents to 0

out_pre, out_post, a = wrapper(x, steer_cfg=steer_cfg)

preds_pre = get_predictions(out_pre, text_features).cpu()
preds_post = get_predictions(out_post, text_features).cpu()

acc_pre = (preds_pre == x["label"]).float().mean()
acc_post = (preds_post == x["label"]).float().mean()

print(acc_pre, acc_post)

If you SAE loaded from Huggingface Hub "hyesulim/fastsae-models" you will see this as the cell output:
> tensor(0.8477) tensor(0.)


See more steering examples in `fastsae-lib/models/sae.py` 

```python
    def steer(self, a: Tensor, steer_cfg: Optional[Dict[str, Any]]) -> Tensor:
        """
        Steering supports:
          steer_cfg = {
            "indices": int | Iterable[int] | Callable[[Tensor], Tensor],
                # which latents to affect; can be:
                #   - int → single index
                #   - iterable of ints → fixed indices
                #   - callable: (a: Tensor) -> LongTensor[batch, k] or [batch]
                #       e.g. lambda x: torch.topk(x, k=3, dim=-1).indices
            # exactly one of the following operations:
            "multiply": float,                  # a[idx] *= multiply
            "add": float,                       # a[idx] += add
            "set": float,                       # a[idx] = set
            "permute": bool,                    # randomly permute activations at given indices
          }
        """
        ...
```

### Classification with ablating class-wise top-k latents

In [ ]:
def get_blocked_latent_indices(intrp):
    assert "cls_wise_top_idx" in intrp.stats
    sorted_items = sorted(intrp.stats["cls_wise_top_idx"].items())
    cls_wise_top_idx_np = np.stack([v.cpu().numpy() for k, v in sorted_items])
    value_counts = np.bincount(cls_wise_top_idx_np.flatten())
    sorted_value_counts = np.sort(value_counts)[::-1]
    sorted_indices = np.argsort(value_counts)[::-1]
    blocked_latent_indices = sorted_indices[sorted_value_counts > 500]
    return torch.tensor(blocked_latent_indices, dtype=torch.long)


def excluded_topk_per_label(labels: torch.Tensor, stats: dict, blocked: torch.Tensor, k: int) -> torch.Tensor:
    # labels: LongTensor [B]
    # stats['cls_wise_top_idx'][label] -> LongTensor [K_stats] (sorted by importance)
    per_label_lists = []
    for y in labels.tolist():
        cls_top = stats["cls_wise_top_idx"][int(y)]
        # filter out any blocked indices, keep ordering
        mask = ~torch.isin(cls_top, blocked.to(cls_top.device))
        filtered = cls_top[mask]
        per_label_lists.append(filtered)

    # ensure consistent width across the batch
    k_eff = min(k, min(t.numel() for t in per_label_lists))
    if k_eff == 0:
        raise ValueError("No available indices after exclusion. Reduce blocked indices or k.")

    return torch.stack([t[:k_eff] for t in per_label_lists], dim=0).to(dtype=torch.long)

In [ ]:
blocked_latent_indices = get_blocked_latent_indices(intrp)
k = 10
indices = excluded_topk_per_label(x["label"], intrp.stats, blocked_latent_indices, k)
steer_cfg = {"indices": indices, "set": 0}

out_pre, out_post, a = wrapper(x, steer_cfg=steer_cfg)

preds_pre = get_predictions(out_pre, text_features).cpu()
preds_post = get_predictions(out_post, text_features).cpu()

acc_pre = (preds_pre == x["label"]).float().mean()
acc_post = (preds_post == x["label"]).float().mean()

print(acc_pre, acc_post)

If you SAE loaded from Huggingface Hub "hyesulim/fastsae-models" you will see this as the cell output:
> tensor(0.8477) tensor(0.3789)

In [ ]:
dataset_instance[0]['image']

In [ ]:
indices

```
tensor([[22139, 19952, 17059,  ...,  2396, 14497, 43424],
        [22139, 19952, 17059,  ...,  2396, 14497, 43424],
        [22139, 19952, 17059,  ...,  2396, 14497, 43424],
        ...,
        [38992, 22956, 13935,  ...,  6079, 34239, 23534],
        [38992, 22956, 13935,  ...,  6079, 34239, 23534],
        [38992, 22956, 13935,  ...,  6079, 34239, 23534]])
```

In [ ]:
intrp.show_ref_samples(indices[0][0])

If you SAE loaded from Huggingface Hub "hyesulim/fastsae-models" you will see this as the cell output:

In [ ]:
PIL.Image.open("output_examples/val_0_22139.png")